# Manga2Motion — Google Colab (T4 GPU) Runner

End-to-end 3-page manga → **30-frame** colorized motion video, tuned for Colab **T4 16GB VRAM** (Tier-2 path: SD1.5 + AnimateDiff + procedural fallbacks).

> **Before you start:** in Colab go to `Runtime → Change runtime type → Hardware accelerator → T4 GPU` → Save. Then run cells **in order Step 0 → Step 8**.

**Numbered steps in this notebook:**
- Step 0 — Verify T4 GPU / CUDA / VRAM
- Step 1 — Get project code + system deps (ffmpeg)
- Step 2 — Install Python deps (T4-optimized)
- Step 3 — T4 memory config (fp16, offload, env vars)
- Step 4 — Upload 3 manga pages (+ optional color reference)
- Step 5 — Smoke test: panel extraction + colorize preview
- Step 6 — Full 30-frame pipeline (CLI)
- Step 7 — Play / download MP4 + timeline.json
- Step 8 — Optional: launch Gradio WebUI
- Step 9 — VRAM cleanup & OOM troubleshooting

## Step 0 — Verify T4 GPU / CUDA / VRAM

In [ ]:
# Step 0: fail fast if you are NOT on a T4 GPU runtime
!nvidia-smi

import torch, sys
print(f"torch={torch.__version__} cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device={torch.cuda.get_device_name(0)}")
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"total_vram_GB={total_gb:.2f}")
    assert total_gb > 10, "Expected ~15GB T4. Check Runtime → T4 GPU."
else:
    print("WARNING: no CUDA. Pipeline will run in slow CPU/procedural fallback mode.")
print(sys.version)

## Step 1 — Get project code + system deps

If you opened this `.ipynb` directly in Colab, the repo files are **not** present yet. The cell below handles both cases:
(a) you uploaded the repo zip, or (b) you clone from GitHub.

In [ ]:
# Step 1: project root + ffmpeg
import os
from pathlib import Path

# --- CONFIG: set this to your GitHub repo if you have one, else leave empty ---
REPO_URL = ""  # e.g. "https://github.com/<you>/manga2motion.git"

if Path("src/run_cli.py").exists() or Path("scripts/run_cli.py").exists():
    print("[Step 1] Already inside project root:", os.getcwd())
elif REPO_URL:
    print(f"[Step 1] Cloning {REPO_URL} ...")
    !git clone $REPO_URL /content/manga2motion
    %cd /content/manga2motion
else:
    print("[Step 1] No src/ found and no REPO_URL set.")
    print("  Option A: upload your project zip via Files pane and unzip it here.")
    print("  Option B: set REPO_URL above and re-run this cell.")
    # Minimal scaffold so later cells fail with a clear message instead of silently
    !pwd && ls -la

# System deps (ffmpeg is required by src/postprocessing/video_encoder.py)
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
!ffmpeg -version | head -n 1

# Persistence dirs expected by Dockerfile / run_cli.py
!mkdir -p inputs outputs models config
!ls -R | head -n 60

## Step 2 — Install Python deps (T4-optimized)

Colab already ships `torch` + CUDA. We install the rest pinned to `numpy<2` (required by `requirements.txt`) and add `xformers` for memory-efficient attention on T4.

In [ ]:
# Step 2: T4-optimized pip install (takes ~3-5 min first time)
# Uninstall incompatible numpy first (Colab defaults to numpy 2.x)
!pip install -q "numpy>=1.24.0,<2.0.0"
!pip install -q "diffusers>=0.35.0" "transformers>=4.44.0" "accelerate>=0.33.0" "safetensors>=0.4.2" "huggingface-hub>=0.24.0"  # >=0.35 for Qwen-Image-Edit / Wan2.1 pipelines
!pip install -q ultralytics>=8.1.0 opencv-python>=4.8.0 pillow>=10.2.0 scipy>=1.11.0 scikit-image>=0.22.0 scikit-learn>=1.3.0
!pip install -q einops>=0.7.0 imageio>=2.34.0 imageio-ffmpeg>=0.4.9 ffmpeg-python>=0.2.0 tqdm>=4.66.0 pyyaml>=6.0.1 pydantic>=2.6.0
!pip install -q gradio>=4.26.0 fastapi>=0.110.0 uvicorn>=0.29.0 python-multipart>=0.0.9 requests>=2.31.0
# Memory-efficient attention for 16GB T4 (optional but strongly recommended)
!pip install -q xformers --no-deps || echo "xformers wheel unavailable, continuing without it"

import diffusers, transformers, accelerate
print("diffusers", diffusers.__version__)
print("transformers", transformers.__version__)
print("accelerate", accelerate.__version__)


## Step 3 — T4 memory config (fp16, offload, env vars)

Maps to `plan.md` **Tier-2 (8–16GB VRAM)**: SD1.5 + AnimateDiff, sequential CPU offload, attention/VAE slicing, 512px working resolution. Heavy diffusion has automatic procedural fallback (`MangaColorizer`, `I2VMotionEngine`) so the notebook never hard-crashes on OOM.

In [ ]:
# Step 3: env vars + T4 yaml patch
import os
os.environ["USE_IMAGE_API"] = "false"          # false = local SD1.5 on T4; true = free Pollinations/HF API offload
os.environ["IMAGE_API_PROVIDER"] = "pollinations"
os.environ["ENABLE_FREE_API_FALLBACK"] = "true"  # auto-fallback if VRAM insufficient
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
os.environ["CUDA_MODULE_LOADING"] = "LAZY"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Optional: paste a free Hugging Face token for higher API rate limits (Colab Secrets recommended)
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("USE_IMAGE_API=", os.environ["USE_IMAGE_API"])
print("ENABLE_FREE_API_FALLBACK=", os.environ["ENABLE_FREE_API_FALLBACK"])

# Patch default_pipeline.yaml for T4 if present (keeps original as .bak)
import shutil
from pathlib import Path
cfg = Path("config/default_pipeline.yaml")
if cfg.exists():
    shutil.copy(cfg, str(cfg) + ".bak")
    txt = cfg.read_text()
    txt = txt.replace('precision: "fp16"', 'precision: "fp16"  # T4-patched')
    # Force 512px working res for 16GB VRAM (768px OOMs with AnimateDiff on T4)
    txt = txt.replace("resolution: [768, 512]", "resolution: [512, 512]")
    txt = txt.replace('base_model: "cagliostrolab/animagine-xl-3.1"', 'base_model: "runwayml/stable-diffusion-v1-5"  # T4: SD1.5 not SDXL')
    cfg.write_text(txt)
    print("[Step 3] Patched config/default_pipeline.yaml for T4 (backup at .bak)")
else:
    print("[Step 3] config/default_pipeline.yaml not found — skipping patch")

# Enable memory savers globally for any diffusers pipeline loaded later
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("[Step 3] T4 memory flags set.")

## Step 4 — Upload 3 manga pages (+ optional color reference)

Upload ~3 consecutive B/W pages (PNG/JPG). Optionally upload one colored reference image for palette locking.

In [ ]:
# Step 4: upload inputs
from google.colab import files
from pathlib import Path
from PIL import Image

print("Upload 3 manga pages (PNG/JPG). Select all 3 at once.")
uploaded = files.upload()  # dict {filename: bytes}

Path("inputs").mkdir(exist_ok=True)
page_paths = []
for name, data in uploaded.items():
    dst = Path("inputs") / name
    dst.write_bytes(data)
    page_paths.append(str(dst))
    try:
        with Image.open(dst) as im:
            print(f"  saved {dst}  size={im.size} mode={im.mode}")
    except Exception as e:
        print(f"  saved {dst} (could not preview: {e})")

page_paths = sorted(page_paths)[:3]
print(f"\n[Step 4] Using pages: {page_paths}")
assert len(page_paths) >= 1, "Upload at least 1 manga page to continue (3 recommended)."

# Optional second upload: color reference (press Skip / cancel if none)
print("\n(Optional) upload ONE color reference image, or skip this by cancelling the picker.")
try:
    ref = files.upload()
    REF_IMAGE = None
    for name, data in ref.items():
        dst = Path("inputs") / ("ref_" + name)
        dst.write_bytes(data)
        REF_IMAGE = str(dst)
        break
    print("REF_IMAGE =", REF_IMAGE)
except Exception as e:
    REF_IMAGE = None
    print("No reference image (", e, ") — palette HEX defaults will be used.")

# Persist for later cells
%store page_paths > /dev/null 2>&1 || True
import json; Path("/tmp/m2m_pages.json").write_text(json.dumps({"pages": page_paths, "ref": REF_IMAGE if 'REF_IMAGE' in dir() else None}))
print("[Step 4] Saved to /tmp/m2m_pages.json")

## Step 5 — Smoke test: panel extraction + colorize preview

Runs `PanelExtractor` → `BubbleDetector` + `MangaInpainter` → `MangaColorizer` (procedural path, no large weights) to confirm your pages work before spending GPU time on the full video.

In [ ]:
# Step 5: layout-preserving colorize preview (no redraw)
# Fix: old mask treated white faces as bubbles (white>220, area<35%) -> SD-inpaint redrew faces.
# Now: inpaint OFF by default; optional text-gated mask; colorize preserves lineart + aspect.
import json, os
from pathlib import Path
import cv2, numpy as np
import matplotlib.pyplot as plt
from PIL import Image

COLOR_ENGINE = "qwen"   # "qwen" | "sd-controlnet" | "procedural"
DO_INPAINT = False      # True only to remove speech text; False = colorize as-is (your ask)
HF_MODEL_EDIT = "Qwen/Qwen-Image-Edit-2509"
HF_TOKEN = os.getenv("HF_TOKEN", "")

meta = json.loads(Path("/tmp/m2m_pages.json").read_text())
def resolve(p):
    for c in [Path(p), Path("/content")/p, Path("/content/manga2motion")/p]:
        if c.exists(): return str(c)
    return p
PAGES = [resolve(p) for p in meta["pages"]]
print("PAGES =", PAGES)

def load_rgb(p):
    bgr = cv2.imread(p, cv2.IMREAD_COLOR)
    assert bgr is not None, f"Not found: {p}"
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def keep_aspect_resize(img, long_side=768):
    h, w = img.shape[:2]
    s = long_side / max(h, w)
    return cv2.resize(img, (int(w*s), int(h*s)), interpolation=cv2.INTER_AREA)

def extract_lineart(rgb):
    # XDoG-ish: keep black inks exact so colorize cannot move faces
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    _, line = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)
    return cv2.cvtColor(line, cv2.COLOR_GRAY2RGB)

def blend_with_lineart(color_rgb, lineart_rgb, alpha=0.85):
    h, w = lineart_rgb.shape[:2]
    color_rgb = cv2.resize(color_rgb, (w, h), interpolation=cv2.INTER_LANCZOS4)
    lnorm = cv2.cvtColor(lineart_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    mult = np.repeat(lnorm[:, :, None], 3, axis=2)
    return np.clip(color_rgb.astype(np.float32) * (1-alpha+alpha*mult), 0, 255).astype(np.uint8)

page = load_rgb(PAGES[0])
print("page shape:", page.shape)
base = keep_aspect_resize(page, 768)   # no square stretch
lineart = extract_lineart(base)

# --- optional text-only inpaint mask (gated: white + has text strokes + oval) ---
clean = base.copy()
if DO_INPAINT:
    gray = cv2.cvtColor(base, cv2.COLOR_RGB2GRAY)
    _, white = cv2.threshold(gray, 225, 255, cv2.THRESH_BINARY)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0); gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1)
    edges = np.uint8(np.clip(cv2.magnitude(gx, gy), 0, 255) > 60) * 255
    n, labels, stats, _ = cv2.connectedComponentsWithStats(white, 8)
    H, W = gray.shape; mask = np.zeros((H, W), np.uint8)
    for i in range(1, n):
        a = stats[i, cv2.CC_STAT_AREA]
        if not (800 < a < H*W*0.08): continue   # faces are >8%; bubbles are small
        comp = (labels == i).astype(np.uint8) * 255
        text_ratio = np.count_nonzero(cv2.bitwise_and(edges, edges, mask=comp)) / float(a)
        cnts, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        sol = a / float(cv2.contourArea(cv2.convexHull(cnts[0])) + 1e-5) if cnts else 0
        if text_ratio > 0.04 and sol > 0.7:   # must look like text balloon
            mask[labels == i] = 255
    print("bubble px:", int((mask > 0).sum()), "/", mask.size)
    if (mask > 0).sum() > 0:
        import torch
        try:
            from diffusers import StableDiffusionInpaintPipeline
            dev = "cuda" if torch.cuda.is_available() else "cpu"
            pipe = StableDiffusionInpaintPipeline.from_pretrained("runwayml/stable-diffusion-inpainting",
                torch_dtype=torch.float16 if dev == "cuda" else torch.float32)
            if dev == "cuda":
                try: pipe.enable_model_cpu_offload()
                except Exception: pipe.enable_attention_slicing()
            h, w = mask.shape
            out = pipe(prompt="plain manga paper texture, empty, no text, no drawing",
                       negative_prompt="face, person, text, bubble, character",
                       image=Image.fromarray(base), mask_image=Image.fromarray(mask),
                       height=(h//8)*8, width=(w//8)*8,
                       num_inference_steps=20, guidance_scale=4.0, strength=0.9).images[0]
            clean = np.array(out.convert("RGB"))
        except Exception as e:
            print("inpaint failed, keeping original:", e)
    lineart = extract_lineart(clean)
else:
    print("[Step 5] inpaint skipped (DO_INPAINT=False) — layout preserved")

# --- colorize (must not redraw) ---
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
colored = None
if COLOR_ENGINE == "qwen":
    try:
        from huggingface_hub import InferenceClient
        client = InferenceClient(model=HF_MODEL_EDIT, token=HF_TOKEN or None)
        prompt = ("Colorize this manga page, exact same composition and faces, keep all black "
                  "ink lines in place, flat anime cel colors, do not redraw or move anything, "
                  "no new text, no watermark.")
        pil_in = Image.fromarray(clean)
        try: out_img = client.image_to_image(prompt=prompt, image=pil_in)
        except TypeError: out_img = client.image_edit(prompt=prompt, image=pil_in)
        raw = out_img.convert("RGB") if isinstance(out_img, Image.Image) else Image.open(out_img).convert("RGB")
        colored = blend_with_lineart(np.array(raw), lineart)  # snap inks back
        print("[Step 5] colorize: Qwen-Edit API + lineart blend")
    except Exception as e:
        print("[Step 5] Qwen failed, fallback sd-controlnet:", e); COLOR_ENGINE = "sd-controlnet"
if COLOR_ENGINE == "sd-controlnet" and colored is None:
    try:
        from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
        cnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_lineart_anime",
            torch_dtype=torch.float16 if device == "cuda" else torch.float32)
        cpipe = StableDiffusionControlNetPipeline.from_pretrained("runwayml/stable-diffusion-v1-5",
            controlnet=cnet, torch_dtype=torch.float16 if device == "cuda" else torch.float32, safety_checker=None)
        cpipe.scheduler = UniPCMultistepScheduler.from_config(cpipe.scheduler.config)
        if device == "cuda":
            try: cpipe.enable_model_cpu_offload()
            except Exception: cpipe.enable_attention_slicing()
        h, w = clean.shape[:2]; W8, H8 = (w//8)*8, (h//8)*8
        cond = cv2.resize(lineart, (W8, H8))
        out = cpipe(prompt="anime cel shading, flat vibrant colors, keep lineart exact, no change to faces",
                    negative_prompt="redraw, different face, distorted, text, watermark, monochrome",
                    image=Image.fromarray(cond), width=W8, height=H8,
                    num_inference_steps=20, guidance_scale=7.0,
                    controlnet_conditioning_scale=1.0).images[0]
        colored = blend_with_lineart(np.array(out.convert("RGB")), lineart)
        print("[Step 5] colorize: SD-ControlNet + lineart blend")
    except Exception as e:
        print("[Step 5] ControlNet failed:", e)
if colored is None:
    g = cv2.cvtColor(clean, cv2.COLOR_RGB2GRAY)
    tiers = np.digitize(g, [70, 140, 210]); c = np.zeros_like(clean, dtype=float)
    c[tiers==0]=np.array([27,38,49])*0.75; c[tiers==1]=np.array([192,57,43])*0.9
    c[tiers==2]=np.array([245,203,167])*0.95; c[tiers==3]=[250,245,235]
    colored = blend_with_lineart(np.clip(c,0,255).astype(np.uint8), lineart)
    print("[Step 5] colorize: procedural + lineart blend")

print("colored shape:", colored.shape)
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(base); ax[0].set_title("original B/W (layout kept)"); ax[0].axis("off")
ax[1].imshow(colored); ax[1].set_title(f"colorized ({COLOR_ENGINE})"); ax[1].axis("off")
plt.tight_layout(); plt.show()



## Step 6 — Full 30-frame pipeline (CLI)

Calls `scripts/run_cli.py` — the same entrypoint as local/Docker — with T4-safe defaults: 30 frames @ 24fps, `dynamic_action_dash` preset, procedural + local-SD1.5 color path. Heavy models (SAM2 / Depth-Anything / AnimateDiff) load only if VRAM allows, otherwise the built-in camera-parallax fallback still renders exactly 30 frames.

In [ ]:
# Step 6: full pipeline — edit prompts / preset here
import json
from pathlib import Path
meta = json.loads(Path("/tmp/m2m_pages.json").read_text())
PAGES_Q = " ".join(f'"{p}"' for p in meta["pages"])
REF = meta.get("ref")
REF_ARG = f'--ref-image "{REF}"' if REF else ""

# --- TUNE THESE ---
MOTION_PRESET = "dynamic_action_dash"  # camera_dolly_in | dynamic_action_dash | impact_freeze_settle | parallax_pan_right | dutch_angle_tilt
MOTION_PROMPT = "rapid forward motion, slashing sword attack, motion blur, glowing blade trail"
TOTAL_FRAMES = 30
FPS = 24

!python scripts/run_cli.py --pages $PAGES_Q --frames $TOTAL_FRAMES --fps $FPS --mode climax_continuity --motion-preset $MOTION_PRESET --motion-prompt "$MOTION_PROMPT" $REF_ARG --output outputs/manga_motion_30f.mp4 --save-timeline outputs/timeline.json

print("\n[Step 6] done — listing outputs:")
!ls -lh outputs/

## Step 7 — Play / download MP4 + timeline.json

In [ ]:
# Step 7: inline playback + download
from IPython.display import Video, display, JSON
from google.colab import files
import json

VIDEO = "outputs/manga_motion_30f.mp4"
TL = "outputs/timeline.json"

print(f"[Step 7] {VIDEO}:")
display(Video(VIDEO, embed=True, width=640))

print(f"\n[Step 7] {TL}:")
display(JSON(json.load(open(TL))))

# Download to your laptop
files.download(VIDEO)
files.download(TL)

## Step 8 — Optional: launch Gradio WebUI

Same UI as `src/ui/app.py` (Docker port 7860), exposed via Colab public link. Uses the same T4 fallbacks, so it stays inside 16GB.

In [ ]:
# Step 8 (optional): interactive UI with timeline editor + palette picker
from src.ui.app import build_gradio_app
app = build_gradio_app()
app.launch(share=True, server_port=7860)  # click the *.gradio.live link

## Step 9 — VRAM cleanup & OOM troubleshooting

| Symptom | Fix |
|---|---|---|
| `CUDA out of memory` in Step 6 | Set `USE_IMAGE_API=true` + `IMAGE_API_PROVIDER=pollinations` and re-run Step 6 (offloads diffusion to free API, keeps panel/depth local) |
| AnimateDiff OOM | It auto-falls-back to camera-parallax — that is expected on T4, output is still exactly 30 frames |
| `numpy` / `scikit-image` binary error | Re-run Step 2 (pins `numpy<2`) then `Runtime → Restart session` and skip back to Step 3 |
| Slow / stalled Gradio | Stop Step 8 cell, run the cleanup below, relaunch |

In [ ]:
# Step 9: free VRAM without restarting the runtime
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"allocated={torch.cuda.memory_allocated()/1e9:.2f}GB reserved={torch.cuda.memory_reserved()/1e9:.2f}GB")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv